# Machine Learning workflow: baseline trước, model sau

## Mục tiêu
- Frame bài toán thành target, feature, prediction time và action.
- Split data trước khi chuẩn bị/đánh giá.
- So sánh baseline với model bằng metric phù hợp.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train.shape, X_test.shape

## Baseline là mốc tối thiểu
Nếu model “xịn” không thắng baseline đơn giản, dừng lại để kiểm tra data/metric/problem. Với classification lệch lớp, accuracy một mình có thể đánh lừa bạn.

In [ ]:
baseline = DummyClassifier(strategy='prior').fit(X_train, y_train)
model = LogisticRegression(max_iter=5_000).fit(X_train, y_train)
baseline_auc = roc_auc_score(y_test, baseline.predict_proba(X_test)[:, 1])
model_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
{'baseline_auc': baseline_auc, 'logistic_auc': model_auc}

In [ ]:
print(classification_report(y_test, model.predict(X_test), target_names=data.target_names))
cv_auc = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
cv_auc.mean(), cv_auc.std()

## Metric theo quyết định
- Fraud/triage: recall cao có thể quan trọng, nhưng cần kiểm soát false positive.
- Marketing: precision/lift ở top-k và giá trị incremental thường quan trọng.
- Medical/safety: không triển khai chỉ dựa trên notebook; cần validation, governance và chuyên gia lĩnh vực.

## Tự luyện
1. Thử `DummyClassifier(strategy='most_frequent')`; tại sao AUC có thể khác?
2. In confusion matrix và chọn threshold khi cost false negative gấp 5 lần false positive.
3. Viết prediction time cho một bài churn bạn tưởng tượng.

**Checkpoint:** Bạn phân biệt được evaluation on test với cross-validation trên training data.